In [1]:
"""
    BM25:关键字字面匹配,靠分词词表匹配打分,通用词(动物,的)会带来大量低分无关文档;
    chroma向量检索:语义相似度匹配,理解句子深层含义,即使无相同词汇,同类语义文本也能召回。
"""
# 配置镜像
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HUB_ENABLE_HF_XET'] = '0'

import shutil
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi
import numpy as np
from typing import List
import hanlp
from transformers import BertTokenizer

def patch_hanlp_tokenizer():
    if not hasattr(BertTokenizer, "encode_plus"):#判断当前BertTokenizer有没有encode_plus方法
        def encode_plus(self, text, **kwargs):
            return self(text, **kwargs)
        BertTokenizer.encode_plus = encode_plus
        print("[补丁] 已为 BertTokenizer 添加 encode_plus 方法")

patch_hanlp_tokenizer()

tokenizer=hanlp.load(hanlp.pretrained.tok.COARSE_ELECTRA_SMALL_ZH)#COARSE粗粒度:实体名词不拆分

documents=[
    "猫咪喜欢在阳光下睡觉，它们每天需要很多睡眠。",
    "金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。",
    "大熊猫主要吃竹子，生活在中国的四川、陕西和甘肃。",
    "海豚是聪明的海洋哺乳动物，它们能用声波定位。",
    "企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。",
    "老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。",
    "鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。",
]

# 解耦分词器传入BM25类
class BM25Retriever:
    """参数corpus 原始文档列表 tok 分词器"""
    def __init__(self,corpus:List[str], tok):
        self.corpus=corpus
        self.tok = tok
        tokenized_corpus=[self.tok(doc) for doc in corpus] #批量把所有文档分词,生成二维词列表
        self.bm25=BM25Okapi(tokenized_corpus) #构建BM25检索索引:统计全局词频、每篇文档词频、文档平均长度、构建倒排索引、用于后续关键字打分。
    
    """ 参数query进行查找,top_k 得分最高的前top_k条"""
    def retrieve(self, query: str, top_k: int = 5) -> List[str]:
        tokenized_query = self.tok(query) #对用户输入查询语句分词
        scores = self.bm25.get_scores(tokenized_query) #计算每一篇文档和查询词的BM25相关性分数
        top_indicices=np.argsort(scores)[::-1][:top_k] #argsort从小到大排序,返回索引 [::-1]从大到小排序索引,[:topk] 取前几个
        return [self.corpus[i] for i in top_indicices]

# 向量模型。加载多语言句与训练模型 将任意中文文本输入,自动转换为固定维度语义向量。
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="paraphrase-multilingual-MiniLM-L12-v2")
# 持久化客户端 
client = chromadb.PersistentClient(path="./chroma_db")#磁盘持久化向量库,程序关闭后向量数据不会丢失。

"""复用集合   先尝试获取名为animal_docs的向量集合,集合不存在则新建,并将全部文档写入向量库,自动生成唯一id
              collection.add 内部自动调用上面的ef嵌入函数,批量生成文档向量存入数据库 """
try:
    collection = client.get_collection(name="animal_docs", embedding_function=ef) 
except Exception:
    collection = client.create_collection(name="animal_docs",embedding_function=ef)
    collection.add(documents=documents,ids=[f"id_{i}" for i in range(len(documents))]) #将ids与document的元素一一对应 id_0： id_1：

# 取出集合全部数据
all_data = collection.get() #get的去有效数据,也可以通过下面query查询数据
print("=== ChromaDB 全部存储数据 ===")
print("所有id列表:", all_data["ids"])
print("\n所有文档内容:")
for doc_id, doc_text in zip(all_data["ids"], all_data["documents"]):
    print(f"{doc_id}: {doc_text}")

#向量检索函数 底层逻辑:自动把query转换为向量,计算库内所有文档向量与query向量的余弦相似度,由高到低排序返回
#当有多个query查询语句，二维向量时候记录多个语句,可以进行批量多查询
def chroma_retrieve(query: str, top_k: int = 5) -> List[str]:
    results = collection.query(query_texts=[query],n_results=top_k) #query_texts:输入查询文本列表。n_results:返回相似度最高的top_k条
    return results['documents'][0]   #results['documents']得到的是二维嵌套列表,取[0]拿到本次查询对应的文档一维列表。

if __name__ == "__main__":  #只有当直接运行当前脚本时候才会执行下面测试代码,被其他文件import导入时不会自动执行。
    query = "聪明的海洋动物"

    print(f"查询语句: {query}\n")
    bm25_retriever = BM25Retriever(documents, tokenizer)
    bm25_results = bm25_retriever.retrieve(query, top_k=5)
    print("BM25 检索结果 (top-5):")
    for idx, doc in enumerate(bm25_results, 1):
        print(f"{idx}. {doc}")

    print("\n" + "-"*40 + "\n")
    chroma_results = chroma_retrieve(query, top_k=5)
    print("ChromaDB 向量检索结果 (top-5):")
    for idx, doc in enumerate(chroma_results, 1):
        print(f"{idx}. {doc}")

c:\Users\26813\.conda\envs\dev2\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


[补丁] 已为 BertTokenizer 添加 encode_plus 方法


[transformers] The following layers were not sharded: encoder.layer.*.attention.self.query.weight, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, embeddings.LayerNorm.bias, encoder.layer.*.output.dense.bias, encoder.layer.*.attention.output.LayerNorm.bias, pooler.dense.weight, embeddings.word_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, embeddings.token_type_embeddings.weight, encoder.layer.*.intermediate.dense.weight, pooler.dense.bias, encoder.layer.*.attention.self.query.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.key.bias, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.attention.output.dense.bias, encoder.layer.*.attention.self.key.weight, embeddings.LayerNorm.weight, embeddings.position_embeddings.weight, encoder.layer.*.attention.self.value.bias


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

=== ChromaDB 全部存储数据 ===
所有id列表: ['id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_5', 'id_6']

所有文档内容:
id_0: 猫咪喜欢在阳光下睡觉，它们每天需要很多睡眠。
id_1: 金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。
id_2: 大熊猫主要吃竹子，生活在中国的四川、陕西和甘肃。
id_3: 海豚是聪明的海洋哺乳动物，它们能用声波定位。
id_4: 企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。
id_5: 老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。
id_6: 鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。
查询语句: 聪明的海洋动物

BM25 检索结果 (top-5):
1. 海豚是聪明的海洋哺乳动物，它们能用声波定位。
2. 老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。
3. 金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。
4. 大熊猫主要吃竹子，生活在中国的四川、陕西和甘肃。
5. 鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。

----------------------------------------

ChromaDB 向量检索结果 (top-5):
1. 海豚是聪明的海洋哺乳动物，它们能用声波定位。
2. 企鹅虽然不会飞，但它们是游泳高手，能在水下快速前进。
3. 鹦鹉可以模仿人类说话，是一种很受欢迎的宠物鸟。
4. 金毛犬是一种非常友好、活泼的犬种，适合家庭饲养。
5. 老虎是大型猫科动物，身上有独特的条纹，是顶级捕食者。
